# 04 Token Classification Pipeline

To process a new file we first need to import the three models and build pipelines for each using the Huggingface [TokenClassificationPipeline](https://huggingface.co/docs/transformers/main_classes/pipelines) 

In [1]:
from transformers import DistilBertForTokenClassification, AutoTokenizer, TokenClassificationPipeline
import torch

In [2]:
from transformers import DistilBertTokenizer

In [3]:
model_i_path = "./model/interventions_classification"

model_i = DistilBertForTokenClassification.from_pretrained(model_i_path)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [5]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [6]:
pipe_i = TokenClassificationPipeline(model = model_i, tokenizer=tokenizer)

In [7]:
sentence = "Assessment of therapeutic response of Plasmodium falciparum to chloroquine and sulfadoxine-pyrimethamine in an area of low malaria transmission in Colombia"

output = pipe_i(sentence)

print(output)

[{'entity': 'B-3', 'score': 0.5690312, 'index': 6, 'word': 'pl', 'start': 38, 'end': 40}, {'entity': 'I-3', 'score': 0.4027279, 'index': 7, 'word': '##as', 'start': 40, 'end': 42}, {'entity': 'I-3', 'score': 0.49906352, 'index': 8, 'word': '##mo', 'start': 42, 'end': 44}, {'entity': 'I-3', 'score': 0.6683265, 'index': 9, 'word': '##dium', 'start': 44, 'end': 48}, {'entity': 'I-3', 'score': 0.72929174, 'index': 10, 'word': 'fa', 'start': 49, 'end': 51}, {'entity': 'I-3', 'score': 0.6153834, 'index': 11, 'word': '##lc', 'start': 51, 'end': 53}, {'entity': 'I-3', 'score': 0.7056071, 'index': 12, 'word': '##ip', 'start': 53, 'end': 55}, {'entity': 'I-3', 'score': 0.6140255, 'index': 13, 'word': '##aru', 'start': 55, 'end': 58}, {'entity': 'I-3', 'score': 0.74561405, 'index': 14, 'word': '##m', 'start': 58, 'end': 59}, {'entity': 'B-3', 'score': 0.7382455, 'index': 16, 'word': 'ch', 'start': 63, 'end': 65}, {'entity': 'B-3', 'score': 0.5238884, 'index': 17, 'word': '##lor', 'start': 65, 'en

In [8]:
output[0]["word"] + "".join(output[1]["word"].replace("##", ""))

'plas'

In [9]:
def reform_words(inputs):
    word_scores_dict = {}

    for i in range(len(inputs)):
        if inputs[i]["word"].startswith("##"):
            continue
        word = inputs[i]["word"]
        for j in range(i+1,len(inputs)):
            if inputs[j]["word"].startswith("##"):
                word += "".join(inputs[j]["word"].replace("##", ""))
            else:
                break
        word_scores_dict[word] = inputs[i]["entity"][-1]
    return word_scores_dict

In [10]:
interventions_output = reform_words(output)
interventions_output

{'plasmodium': '3',
 'falciparum': '3',
 'chloroquine': '3',
 'and': '3',
 'sulfadoxine': '3',
 '-': '3',
 'pyrimethamine': '3'}

In [11]:
thingy = ['Assessment of therapeutic response of Plasmodium falciparum to chloroquine and sulfadoxine-pyrimethamine in an area of low malaria transmission in Colombia.',
 '',
 'Although chloroquine (CQ) resistance was first reported in Colombia in 1961 and sulfadoxine-pyrimethamine (SP) resistance in 1981, the frequency of treatment failures to these drugs in Colombia is unclear. A modified World Health Organization 14-day in vivo drug efficacy test for uncomplicated Plasmodium falciparum malaria in areas with intense malaria transmission was adapted to reflect the clinical and epidemiologic features of a low-intensity malaria transmission area in the Pacific Coast Region of Colombia. Patients > or =1 year of age with a parasite density > or =1,000 asexual parasites per microliter were enrolled in this study. Forty-four percent (24 of 54) of the CQ-treated patients were therapeutic failures, including 7 early treatment failures (ETFs) and 17 late treatment failures (LTFs). Four (6%) of 67 SP-treated patients were therapeutic failures (2 ETFs and 2 LTFs). Therapeutic failure in the CQ-treated group was associated with an age <15 years old (P < 0.01), but was not associated with initial parasite density, the presence of CQ or sulfa-containing drugs in urine, or a history of malaria. The high level of therapeutic failures to CQ detected in this study underscores the need and importance of drug efficacy evaluation in the development of a rational national antimalarial drug policy. The relatively low level of therapeutic failures to SP compared with other South American countries raises further questions regarding factors that might have prevented the rapid development of in vivo resistance to this drug combination.',
 '']

In [12]:
file_output = pipe_i(thingy)

file_output_tokens = {}
for i in range(len(file_output)):
    output2 = reform_words(file_output[i])
    for k, v in output2.items():
        if k not in file_output_tokens:
            file_output_tokens[k] = v


file_output_tokens

{'plasmodium': '3',
 'falciparum': '3',
 'chloroquine': '3',
 'and': '3',
 'sulfadoxine': '3',
 '-': '3',
 'pyrimethamine': '3',
 'chquine': '3',
 '(': '3',
 'cq': '3',
 ')': '3',
 'sp': '3',
 'c': '3'}

In [13]:
model_p_path = "./participants_classification_model"

model_p = DistilBertForTokenClassification.from_pretrained(model_p_path)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [14]:
pipe_p = TokenClassificationPipeline(model=model_p, tokenizer=tokenizer)

In [15]:
output_p = pipe_p(sentence)

print(output_p)

[{'entity': 'I-4', 'score': 0.4860548, 'index': 8, 'word': '##mo', 'start': 42, 'end': 44}, {'entity': 'I-4', 'score': 0.7236243, 'index': 10, 'word': 'fa', 'start': 49, 'end': 51}, {'entity': 'I-4', 'score': 0.6238239, 'index': 11, 'word': '##lc', 'start': 51, 'end': 53}, {'entity': 'I-4', 'score': 0.5859724, 'index': 12, 'word': '##ip', 'start': 53, 'end': 55}, {'entity': 'I-4', 'score': 0.5968856, 'index': 13, 'word': '##aru', 'start': 55, 'end': 58}, {'entity': 'I-4', 'score': 0.50851613, 'index': 14, 'word': '##m', 'start': 58, 'end': 59}, {'entity': 'I-4', 'score': 0.38621765, 'index': 37, 'word': 'low', 'start': 119, 'end': 122}, {'entity': 'I-4', 'score': 0.74670064, 'index': 38, 'word': 'malaria', 'start': 123, 'end': 130}, {'entity': 'I-4', 'score': 0.7250304, 'index': 39, 'word': 'transmission', 'start': 131, 'end': 143}, {'entity': 'I-4', 'score': 0.47537366, 'index': 41, 'word': 'colombia', 'start': 147, 'end': 155}]


In [16]:
participants_output = reform_words(output_p)
participants_output

{'falciparum': '4',
 'low': '4',
 'malaria': '4',
 'transmission': '4',
 'colombia': '4'}

In [17]:
model_o_path = "./outcomes_classification_model"

model_o = DistilBertForTokenClassification.from_pretrained(model_o_path)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [18]:
pipe_o = TokenClassificationPipeline(model=model_o, tokenizer=tokenizer)

In [19]:
output_o = pipe_o(sentence)

print(output_o)

[]


In [22]:
outcomes_output = reform_words(output_o)

outcomes_output

{}

In [20]:
import pandas as pd

In [23]:
interventions_list = []
participants_list = []
outcomes_list = []

for k, v in interventions_output.items():
    data_tuple = (k, "intervention", v)
    interventions_list.append(data_tuple)

for k, v in participants_output.items():
    data_tuple = (k, "participants", v)
    participants_list.append(data_tuple)

for k, v in outcomes_output.items():
    data_tuple = (k, "participants", v)
    outcomes_list.append(data_tuple)

total_data = interventions_list + participants_list + outcomes_list

int_df = pd.DataFrame(total_data)

int_df

,0,1,2
0,plasmodium,intervention,3
1,falciparum,intervention,3
2,chloroquine,intervention,3
3,and,intervention,3
4,sulfadoxine,intervention,3
5,-,intervention,3
6,pyrimethamine,intervention,3
7,falciparum,participants,4
8,low,participants,4
9,malaria,participants,4


In [27]:
int_df.rename(columns={0:"token", 1:"classification", 2:"label"})

,token,classification,label
0,plasmodium,intervention,3
1,falciparum,intervention,3
2,chloroquine,intervention,3
3,and,intervention,3
4,sulfadoxine,intervention,3
5,-,intervention,3
6,pyrimethamine,intervention,3
7,falciparum,participants,4
8,low,participants,4
9,malaria,participants,4
